# Ingest — organisation data from Microsoft Entra ID

Reads every user in Microsoft Entra ID through Microsoft Graph and writes `copilot_org_data`: one row
per person with department, job title, company, location and manager. Nothing to export or upload,
and it runs unattended with an app registration.

**When to use it.** By default the template takes employee attributes from the Viva query. Run this
notebook when that query carries few of them, or none. The template prefers `copilot_org_data` over
`org_attributes`, keeps only the people on the licensed roster, and lets Viva's attributes fill any
gaps.

**Source:** `/v1.0/users?$select=...&$expand=manager($select=userPrincipalName)&$top=999`, which
pages through every user and resolves each manager's UPN inline.

**Permissions:** an app registration with the `User.Read.All` application permission,
admin-consented. Keep its client secret in Azure Key Vault.

**One writer per table.** Each run replaces `copilot_org_data`. If another notebook already writes
that table in this Lakehouse, run only one of them.


## 1. Configuration

In [ ]:
# === CONFIG ===
# Keep the secret in Key Vault rather than here:
#   CLIENT_SECRET = notebookutils.credentials.getSecret("https://<vault>.vault.azure.net/", "<secret-name>")
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

OUTPUT_TABLE  = 'copilot_org_data'
WRITE_MODE    = 'overwrite'
ALLOW_EMPTY_SNAPSHOT = False   # set True only for an intentional empty first install


## 2. Authenticate to Microsoft Graph

In [ ]:
import requests

def get_graph_token(tenant_id, client_id, client_secret):
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_unset = [name for name, value in (('TENANT_ID', TENANT_ID), ('CLIENT_ID', CLIENT_ID), ('CLIENT_SECRET', CLIENT_SECRET))
          if not str(value or '').strip() or str(value).strip().startswith('<')]
if _unset:
    raise ValueError(f"Set {', '.join(_unset)} in the configuration cell before running this notebook.")

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}'}
print('✓ Graph token acquired.')

import time

GRAPH_MAX_ATTEMPTS = 6     # one try plus up to five retries
GRAPH_MAX_WAIT_SEC = 120   # longest single wait, even if Graph asks for more


def _graph_wait_seconds(response, attempt):
    try:
        wait = float(response.headers.get('Retry-After', ''))
    except (TypeError, ValueError):
        wait = 5 * 2 ** attempt
    return min(max(wait, 1.0), GRAPH_MAX_WAIT_SEC)


def graph_get(url, timeout=60):
    """GET from Graph. Waits out throttling (429) and brief outages (5xx, dropped connections),
    and gets a new token once if the old one expires partway through a long pull."""
    global token, headers
    renewed = False
    for attempt in range(1, GRAPH_MAX_ATTEMPTS + 1):
        last = attempt == GRAPH_MAX_ATTEMPTS
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
        except (requests.ConnectionError, requests.Timeout) as exc:
            if last:
                raise
            wait = min(5 * 2 ** attempt, GRAPH_MAX_WAIT_SEC)
            print(f'  Graph request failed ({type(exc).__name__}); trying again in {wait}s.')
            time.sleep(wait)
            continue
        if r.status_code == 401 and not renewed and not last:
            token = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
            headers = {'Authorization': f'Bearer {token}'}
            renewed = True
            continue
        if (r.status_code == 429 or r.status_code >= 500) and not last:
            wait = _graph_wait_seconds(r, attempt)
            print(f'  Graph returned {r.status_code}; trying again in {wait:.0f}s.')
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r


## 3. Page through `/users` with manager expand

Graph caps `$top` at 999. Follow `@odata.nextLink` until exhausted. Each page goes through `graph_get`, which waits out throttling and brief outages, so a large tenant's long pull doesn't fail on one busy page.


In [ ]:
select_fields  = 'userPrincipalName,displayName,department,jobTitle,companyName,officeLocation,city,country,accountEnabled'
expand_manager = 'manager($select=userPrincipalName)'
next_url = f'https://graph.microsoft.com/v1.0/users?$select={select_fields}&$expand={expand_manager}&$top=999'


def _validate_users_page(data, page_number):
    if not isinstance(data, dict):
        raise ValueError(f'Graph /users page {page_number} did not return an object.')
    if 'value' not in data:
        raise ValueError(f"Graph /users page {page_number} is missing required 'value'; refusing to overwrite org data.")
    value = data['value']
    if not isinstance(value, list):
        raise ValueError(f"Graph /users page {page_number} returned a non-list 'value'; refusing to overwrite org data.")
    next_link = data.get('@odata.nextLink')
    if next_link is not None and (not isinstance(next_link, str) or not next_link.strip()):
        raise ValueError(f"Graph /users page {page_number} returned an invalid '@odata.nextLink'.")
    for item_number, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            raise ValueError(f'Graph /users page {page_number} item {item_number} is not an object.')
    return value, next_link


users = []
seen_urls = set()
page_number = 0
while next_url:
    if next_url in seen_urls:
        raise ValueError(f'Graph /users paging loop detected at page {page_number + 1}.')
    seen_urls.add(next_url)
    page_number += 1
    r = graph_get(next_url, timeout=60)
    page, next_url = _validate_users_page(r.json(), page_number)
    users.extend(page)
    print(f'  Fetched {len(users):,} users so far...')

print(f'✓ Total users: {len(users):,}')


## 4. Flatten manager UPN and name the columns

`manager` is a nested object, so it is flattened to `managerUPN`. Graph's `department` is written as
`Department`, the column the template's Org table reads, `userPrincipalName` as `PersonId` and
`jobTitle` as `JobTitle`. The other Graph names (`displayName`, `companyName`, `officeLocation`,
`city`, `country`) already match the template's aliases. `accountEnabled` has no fixed column, so the
report offers it as an extra attribute.


In [ ]:
def _normalise_identity(value):
    return (value or '').strip().lower()


def _canonical_org_row(user):
    if not isinstance(user, dict):
        raise ValueError('Org data item is not an object.')
    norm = _normalise_identity(user.get('userPrincipalName'))
    if not norm:
        raise ValueError('Org data row is missing userPrincipalName; refusing to write an anonymous identity.')
    manager = user.get('manager')
    manager_upn = ''
    if manager not in (None, ''):
        if not isinstance(manager, dict):
            raise ValueError(f'Org data row {norm!r} has a non-object manager payload.')
        manager_upn = manager.get('userPrincipalName', '') or ''
    return {
        'PersonId':         user.get('userPrincipalName'),
        'displayName':      user.get('displayName'),
        'Department':       user.get('department'),
        'JobTitle':         user.get('jobTitle'),
        'companyName':      user.get('companyName'),
        'officeLocation':   user.get('officeLocation'),
        'city':             user.get('city'),
        'country':          user.get('country'),
        'accountEnabled':   str(user.get('accountEnabled', '')),
        'managerUPN':       manager_upn,
    }


def _dedupe_org_rows(rows):
    seen = {}
    deduped = []
    for row in rows:
        norm = _normalise_identity(row.get('PersonId'))
        comparable = {key: '' if value is None else str(value).strip() for key, value in row.items()}
        prior = seen.get(norm)
        if prior is None:
            seen[norm] = comparable
            deduped.append(row)
            continue
        if comparable != prior:
            raise ValueError(f'Conflicting org rows detected for {norm!r}; refusing to overwrite a good snapshot.')
    return deduped


rows = _dedupe_org_rows([_canonical_org_row(user) for user in users])
print(f'Built {len(rows):,} canonicalised rows.')


# How many people have a department. Views by department leave out everyone without one.
LOW_DEPARTMENT_COVERAGE = 0.8


def department_coverage(departments):
    """(people with a department, people) for the `Department` values of an org table."""
    values = list(departments)
    have = sum(1 for v in values if v is not None and str(v).strip())
    return have, len(values)


def coverage_message(have, total):
    """One line on department coverage, with advice when views by department would leave many out."""
    line = f'{have:,} of {total:,} people have a department'
    if not total or have / total >= LOW_DEPARTMENT_COVERAGE:
        return line
    return (f'{line}. Views by department leave out everyone without one: fill in Department on users '
            'in Microsoft Entra ID.')


_have_dept, _people = department_coverage(row.get('Department') for row in rows)
print(('✓ ' if not _people or _have_dept / _people >= LOW_DEPARTMENT_COVERAGE else '⚠ ')
      + coverage_message(_have_dept, _people))


## 5. Build Spark DataFrame + add normalised key + sanitise columns

Uses an **explicit schema** because Graph's `/users` response can have all-`None` values for some columns on small tenants (e.g. nobody has a `country` set), and Spark's automatic schema inference fails on those with `[CANNOT_DETERMINE_TYPE]`.


In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


def _guard_org_snapshot(row_count, table_name):
    exists = _table_exists(table_name)
    if row_count == 0 and exists:
        raise ValueError(f'Fetched 0 org rows; refusing to replace existing {table_name}.')
    if row_count == 0 and not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError(
            f'Fetched 0 org rows and {table_name} does not exist yet. '
            'Set ALLOW_EMPTY_SNAPSHOT = True only for an intentional empty first install.'
        )


schema = StructType([
    StructField('PersonId',        StringType(), True),
    StructField('displayName',     StringType(), True),
    StructField('Department',      StringType(), True),
    StructField('JobTitle',        StringType(), True),
    StructField('companyName',     StringType(), True),
    StructField('officeLocation',  StringType(), True),
    StructField('city',            StringType(), True),
    StructField('country',         StringType(), True),
    StructField('accountEnabled',  StringType(), True),
    StructField('managerUPN',      StringType(), True),
])

df = spark.createDataFrame(rows, schema=schema)
df = df.withColumn(
    'PersonId_Normalized',
    F.when(F.col('PersonId').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('PersonId').cast('string'))))
)

duplicate_people = (
    df.filter(F.col('PersonId_Normalized').isNotNull() & (F.col('PersonId_Normalized') != ''))
      .groupBy('PersonId_Normalized').count()
      .filter(F.col('count') > 1)
      .count()
)
if duplicate_people:
    raise ValueError('Org data still contains duplicate PersonId_Normalized values after validation.')

total = df.count()
_guard_org_snapshot(total, OUTPUT_TABLE)

_INVALID = re.compile(r'[ ,;{}()\n\t=]')
df = df.toDF(*[_INVALID.sub('_', c) for c in df.columns])

print(f'Final rows: {total:,}')
print('Columns:', df.columns)


## 6. Write to the Lakehouse

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 7. Verify

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
tbl.select('PersonId', 'PersonId_Normalized', 'Department', 'JobTitle', 'managerUPN').show(10, truncate=False)
tbl.groupBy('Department').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**In the report:** refresh the semantic model after this notebook runs. The Fabric template reads
`copilot_org_data` ahead of `org_attributes`, keeps the people on the licensed roster, and takes
Department, Job Title, City, Country, Business Unit (`companyName`) and Manager from it. Viva's own
attributes fill any gaps. `accountEnabled` appears as an extra attribute, which shows licences still
held by disabled accounts.
